In [1]:
import numpy as np
import scipy.stats as st

class BinomialResearchPipeline:
    def __init__(self, n_trials: int = 100, alpha: float = 0.05):
        self.n = n_trials
        self.alpha = alpha

    def get_critical_interval(self, p_null: float) -> tuple:
        """Вычисляет 95% критический интервал для числа успехов при нулевой гипотезе."""
        # Двусторонняя критическая зона
        lower = st.binom.ppf(self.alpha / 2, self.n, p_null)
        upper = st.binom.ppf(1 - self.alpha / 2, self.n, p_null)
        return int(lower), int(upper)

    def run_hypothesis_test(self, successes: int, p_null: float) -> float:
        """Выполняет точный биномиальный критерий (замена устаревшего binom_test)."""
        # Изучаем распределение с помощью актуального scipy API
        result = st.binomtest(successes, n=self.n, p=p_null, alternative='two-sided')
        p_value = result.pvalue

        status = "Отклоняется" if p_value < self.alpha else "Не отклоняется"
        print(f"[TEST] Успехов: {successes}/{self.n} | H0 (p={p_null:.3f}) -> {status} (p-value: {p_value:.5f})")
        return p_value

    def compute_true_power(self, p_null: float, p_alt: float) -> float:
        """
        Математически верное вычисление мощности критерия (1 - Beta).
        Вероятность обнаружить эффект (отклонить H0), если верна альтернативная гипотеза H1.
        """
        # 1. Находим критические границы в терминах количества успехов для H0
        lower_crit = st.binom.ppf(self.alpha / 2, self.n, p_null)
        upper_crit = st.binom.ppf(1 - self.alpha / 2, self.n, p_null)

        # 2. Мощность — это вероятность попасть вне этих границ, ЕСЛИ распределение сместилось к p_alt
        # P(X <= lower_crit | H1) + P(X > upper_crit | H1)
        type_2_error = st.binom.cdf(upper_crit, self.n, p_alt) - st.binom.cdf(lower_crit - 1, self.n, p_alt)
        true_power = 1 - type_2_error
        return float(true_power)


if __name__ == "__main__":
    pipeline = BinomialResearchPipeline(n_trials=100, alpha=0.05)

    print("=== ЭКСПЕРИМЕНТ №1 (20 голов из 100) ===")
    p_null_1 = 1/5  # 0.20
    p_alt_1 = 1/7   # ~0.14

    low, high = pipeline.get_critical_interval(p_null_1)
    print(f"[INFO] Границы нормы для H0 (p=1/5): от {low} до {high} голов")

    # Запустим тест для 20 успехов
    pipeline.run_hypothesis_test(successes=20, p_null=p_null_1)

    # Считаем честную мощность
    power_1 = pipeline.compute_true_power(p_null=p_null_1, p_alt=p_alt_1)
    print(f"[MATH] Честная мощность критерия против альтернативы {p_alt_1:.3f}: {power_1:.4f}\n")

    print("=== ЭКСПЕРИМЕНТ №2 (30 голов из 100) ===")
    p_null_2 = 1/5  # 0.20
    p_alt_2 = 1/3   # ~0.33

    pipeline.run_hypothesis_test(successes=30, p_null=p_null_2)
    power_2 = pipeline.compute_true_power(p_null=p_null_2, p_alt=p_alt_2)
    print(f"[MATH] Честная мощность критерия против альтернативы {p_alt_2:.3f}: {power_2:.4f}")

=== ЭКСПЕРИМЕНТ №1 (20 голов из 100) ===
[INFO] Границы нормы для H0 (p=1/5): от 12 до 28 голов
[TEST] Успехов: 20/100 | H0 (p=0.200) -> Не отклоняется (p-value: 1.00000)
[MATH] Честная мощность критерия против альтернативы 0.143: 0.2168

=== ЭКСПЕРИМЕНТ №2 (30 голов из 100) ===
[TEST] Успехов: 30/100 | H0 (p=0.200) -> Отклоняется (p-value: 0.01695)
[MATH] Честная мощность критерия против альтернативы 0.333: 0.8476
